# Chapter 8: Knowledge Distillation — Making Powerful Models Practical

This chapter covers:
- Distilling knowledge from large models
- Using temperature-scaled soft targets
- Building DeepSeek-R1's distilled models

In this chapter, we take everything our massive DeepSeek-R1 model has learned and compress it into models small enough to run on a single consumer GPU or even a phone. The key technique is **knowledge distillation** — training a small "student" model to reproduce the behavior of a larger "teacher" model.

## Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

---

## The Teacher-Student Paradigm

Instead of training a small model from scratch on raw data, we train it to **mimic a larger, more capable model**. The large model becomes the **teacher**, and the small model becomes the **student**.

### Why Soft Labels Carry More Information Than Hard Labels

A **hard label** is a one-hot vector: `[1, 0, 0, 0]` — approximately 1 bit of information per sample.

A **soft label** from the teacher: `[cat: 86%, dog: 12%, fox: 1.5%, bird: 0.5%]` — communicates a rich web of inter-class relationships. This is what Geoffrey Hinton called **"dark knowledge"** in his seminal 2015 paper.

---

## Temperature and Dark Knowledge

Under a standard softmax (T=1), the teacher's output is too peaked — dark knowledge is hidden in near-zero probabilities. **Temperature scaling** solves this by dividing logits by T before softmax:

$$q_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **T=1**: Standard softmax (too peaked)
- **T=2–5**: The Goldilocks zone (dark knowledge emerges)
- **T≥10**: Too soft (signal lost in near-uniformity)

### Listing 8.1: Visualizing Temperature-Scaled Softmax

In [ ]:
def visualize_temperature(logits, temps):
    """Show how temperature affects softmax."""
    for T in temps:
        probs = F.softmax(logits / T, dim=-1)
        print(f"T={T:2d}: {[f'{p:.3f}' for p in probs.tolist()]}")

logits = torch.tensor([5.0, 3.0, 1.0, -1.0])
visualize_temperature(logits, [1, 2, 3, 5, 10])